In [2]:
!python -m pip install transformers datasets tiktoken pandas tqdm --quiet

In [ ]:
import json
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer

CSV_PATH = "/uufs/chpc.utah.edu/common/home/u1528744/interpretability/cs6966-project/local_datasets/test_set_OOD_megtong.csv"

OUT_DIR = Path("/uufs/chpc.utah.edu/common/home/u1528744/interpretability/cs6966-project/local_datasets/crosscoder_corpus")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV   = OUT_DIR / "crosscoder_eval_megtong_chattemplate_32tok.csv"
OUT_JSONL = OUT_DIR / "crosscoder_eval_megtong_chattemplate_32tok.jsonl"

TOKENIZER_ID = "google/gemma-2-2b-it"   # use same template logic as crosscoder training
MAX_RESP_TOKENS = 32

df = pd.read_csv(CSV_PATH)
for col in ["prompt", "chosen", "rejected"]:
    if col not in df.columns:
        raise ValueError(f"Missing column {col}; found {list(df.columns)}")

if "template_type" not in df.columns:
    df["template_type"] = ""
if "source" not in df.columns:
    df["source"] = ""

tok = AutoTokenizer.from_pretrained(TOKENIZER_ID, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def trunc_k_tokens(text: str, k: int) -> str:
    ids = tok(text, add_special_tokens=False)["input_ids"][:k]
    return tok.decode(ids, skip_special_tokens=True)

rows = []
for i, r in df.iterrows():
    prompt = str(r["prompt"])

    for field in ["chosen", "rejected"]:
        resp_full = str(r[field])
        resp_tr   = trunc_k_tokens(resp_full, MAX_RESP_TOKENS)

        conversation = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": resp_tr},
        ]

        text = tok.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=False,
        )

        rows.append({
            "eval_id": f"{i}_{field}",
            "source_row": int(i),
            "response_field": field,              # keep chosen/rejected as-is
            "template_type": str(r["template_type"]),
            "source": str(r["source"]),
            "prompt": prompt,
            "response_trunc": resp_tr,
            "text": text,
        })

eval_df = pd.DataFrame(rows)
eval_df.to_csv(OUT_CSV, index=False)

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for rec in eval_df.to_dict(orient="records"):
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("Wrote:", OUT_CSV)
print("Wrote:", OUT_JSONL)
print("Eval sequences:", len(eval_df), "(= 2 * original rows)")

ImportError: apply_chat_template requires jinja2 to be installed. Please install it using `pip install jinja2`.